
# Yambda 10 · Final test: SASRec (Colab GPU)

**Run once.** The test split (train < day 287, evaluate days 287–301) has never been used for any
decision. Hyperparameters and the number of epochs come from validation (`best_sasrec.json`, v1:
best epoch 12). No early stopping on test: that would be choosing the epoch by looking at test.

Needs `colab_bundle.zip` and `colab_src.zip` in `MyDrive/music_recsys/`. Runtime → T4 GPU. ~10 min.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/music_recsys"
!rm -rf /content/music_recsys && mkdir -p /content/music_recsys
!unzip -q "{DRIVE}/colab_bundle.zip" -d /content/music_recsys
!unzip -q -o "{DRIVE}/colab_src.zip" -d /content/music_recsys
!mkdir -p /content/music_recsys/reports/yambda && cp "{DRIVE}/outputs/best_sasrec.json" /content/music_recsys/reports/yambda/

import sys
sys.path.insert(0, "/content/music_recsys/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime!)")


In [ ]:

import json, time
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate, artist_novelty, ArtistLens, known_artist_share
from musicrec.yambda import load_item_artist
from musicrec.models.sasrec import SASRec

OUT = config.PROJECT_ROOT / "reports" / "yambda"
split = Split.load(config.YAMBDA_SPLIT_TEST_DIR)
targets_all = pd.read_parquet(config.YAMBDA_SPLIT_TEST_DIR / "targets_all.parquet")
v1 = json.loads((OUT / "best_sasrec.json").read_text())
print(split.stats)
print("params:", v1["params"], "| epochs:", v1["best_epoch"])


In [ ]:

sas = SASRec(**v1["params"], epochs=v1["best_epoch"], early_stopping=False)
t = time.time(); sas.fit(split); print(f"trained in {time.time() - t:.0f}s")


In [ ]:

item_artist = load_item_artist()
lens = ArtistLens(split, item_artist)
new_artist = artist_novelty(split, item_artist)

main = evaluate(sas, split, return_per_user=True)
result = {k: v for k, v in main.items() if k != "per_user"}
result["recall@10 | all targets"] = evaluate(sas, split, targets=targets_all)["recall@10"]
result["recall@10 | known-artist"] = evaluate(sas, split, targets=split.targets[~new_artist])["recall@10"]
result["recall@10 | new-artist"] = evaluate(sas, split, targets=split.targets[new_artist])["recall@10"]
result["known-artist slots in top-10"] = known_artist_share(sas, split, lens)
result


In [ ]:

main["per_user"].to_parquet(OUT / "per_user_test_sasrec.parquet", index=False)
(OUT / "test_sasrec.json").write_text(json.dumps({"name": sas.name, "params": sas.params(),
                                                 "epochs": v1["best_epoch"], "test": result}, indent=2, default=float))
!mkdir -p "{DRIVE}/outputs_test" && cp {OUT}/per_user_test_sasrec.parquet {OUT}/test_sasrec.json "{DRIVE}/outputs_test/"
!ls -la "{DRIVE}/outputs_test"
